In [1]:
# =========================
# LIMPIEZA DE ENTORNO
# =========================

!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu protobuf transformers peft accelerate -q

# =========================
# INSTALACIÓN BASE (COMPATIBLE)
# =========================

!pip install transformers==4.38.2 accelerate==0.27.2 protobuf==3.20.3 -q

# =========================
# MÉTRICAS
# =========================

!pip install evaluate bert-score sacrebleu unbabel-comet bert-score -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 3.20.3 which is incompatible.
google-cloud-datastore 2.25.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataplex 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-api-core 2.30.3 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-bigquery-storage 2.39.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-dataproc 5.28.0 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires

In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [3]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

In [4]:
#Aplicamos limpieza

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

# SILABIFICADOR

In [5]:
# =========================
# DEFINICIÓN DEL SISTEMA
# =========================

VOWELS = ["a", "e", "i", "o"]
DIGRAPHS = ["ch", "sh", "ts", "ty"]
NASALS = ["n", "m"]


# =========================
# TOKENIZADOR
# =========================

def tokenize(word):
    """
    Regla: los dígrafos son una sola unidad fonológica
    """
    tokens = []
    i = 0

    while i < len(word):
        if i + 1 < len(word) and word[i:i+2] in DIGRAPHS:
            tokens.append(word[i:i+2])
            i += 2
        else:
            tokens.append(word[i])
            i += 1

    return tokens


# =========================
# SILABIFICADOR V3.1
# =========================

def silabificar_v3(word):
    tokens = tokenize(word)
    syllables = []
    i = 0

    while i < len(tokens):
        syllable = ""

        # -------------------------
        # Regla: inicio consonántico opcional
        # -------------------------
        if tokens[i] not in VOWELS:
            syllable += tokens[i]
            i += 1

        # -------------------------
        # Regla: núcleo vocálico obligatorio
        # -------------------------
        if i < len(tokens) and tokens[i] in VOWELS:
            syllable += tokens[i]
            i += 1

            #  Regla 9: agrupar vocales consecutivas (diptongos)
            while i < len(tokens) and tokens[i] in VOWELS:
                syllable += tokens[i]
                i += 1

        else:
            syllables.append(syllable)
            break

        # -------------------------
        # Regla: coda nasal opcional (n o m)
        # -------------------------
        if i < len(tokens) and tokens[i] in NASALS:
            if i + 1 < len(tokens) and tokens[i+1] not in VOWELS:
                syllable += tokens[i]
                i += 1

        syllables.append(syllable)

    # =========================
    # POST-PROCESAMIENTO
    # =========================

    # -------------------------
    # Regla 6: evitar nasal final en palabra
    # -------------------------
    if len(syllables) > 1:
        last = syllables[-1]

        if last in NASALS:
            syllables[-2] += last
            syllables = syllables[:-1]

    # -------------------------
    # Regla 7: evitar sílabas sin vocal
    # -------------------------
    cleaned = []
    for syl in syllables:
        if any(v in syl for v in VOWELS):
            cleaned.append(syl)
        else:
            if cleaned:
                cleaned[-1] += syl
            else:
                cleaned.append(syl)

    return cleaned


# =========================
# TEST
# =========================

words = [
    "maranke",
    "ampeji",
    "inki",
    "inchato",
    "wishawe",
    "ratekashamaira",
    "wanoparirin"
]

for w in words:
    print(w, "→", silabificar_v3(w))

maranke → ['ma', 'ran', 'ke']
ampeji → ['am', 'pe', 'ji']
inki → ['in', 'ki']
inchato → ['in', 'cha', 'to']
wishawe → ['wi', 'sha', 'we']
ratekashamaira → ['ra', 'te', 'ka', 'sha', 'mai', 'ra']
wanoparirin → ['wa', 'no', 'pa', 'ri', 'rin']


In [6]:
def silabificar_texto(texto):
    """
    Regla: el silabificador trabaja a nivel de palabra,
    por lo que primero se separa la frase en palabras.
    """
    palabras = texto.split()
    resultado = []

    for palabra in palabras:
        silabas = silabificar_v3(palabra)
        resultado.extend(silabas)

    return resultado

In [10]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificar_v3(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))



# ==============================================================================
# FILTRAR TOKENS ÚTILES DEL SILABIFICADOR (UMBRAL >= 20)
# ==============================================================================
nuevos_tokens = [token for token, count in subword_counter.items() if count >= 20]

print("Cantidad de nuevos tokens a agregar (freq >= 20):", len(nuevos_tokens))


SUBWORDS MÁS FRECUENTES:
[('ra', 2860), ('ki', 2243), ('ke', 2043), ('ja', 2017), ('ti', 2000), ('i', 1866), ('a', 1580), ('we', 1426), ('ka', 1389), ('ri', 1276), ('ma', 1165), ('na', 1141), ('mia', 1035), ('ea', 940), ('en', 794), ('yo', 759), ('ya', 711), ('ba', 621), ('ko', 592), ('o', 555), ('ta', 545), ('pa', 535), ('min', 506), ('bi', 505), ('no', 487), ('to', 478), ('ni', 451), ('nai', 397), ('bo', 327), ('kon', 315), ('nan', 312), ('kai', 310), ('be', 295), ('rin', 284), ('ne', 276), ('sai', 272), ('pi', 269), ('noa', 246), ('tom', 243), ('kin', 243), ('me', 241), ('ro', 230), ('kan', 229), ('shi', 225), ('nin', 224), ('pan', 220), ('man', 220), ('te', 218), ('yoi', 206), ('non', 205)]
Cantidad de nuevos tokens a agregar (freq >= 20): 146


In [11]:
for i in range(5):
    print("ORIGINAL:", df.iloc[i]["source"])
    print("LIMPIO:", df.iloc[i]["source_clean"])

    print("TARGET:", df.iloc[i]["target_clean"])
    print("-----")

ORIGINAL: jatian miabicho ika, ¿ikon?
LIMPIO: jatian miabicho ika ikon
TARGET: entonces estabas solo verdad
-----
ORIGINAL: enra min yoiyai keskaribi shinanai
LIMPIO: enra min yoiyai keskaribi shinanai
TARGET: concuerdo contigo
-----
ORIGINAL: tomra jabicho ibake
LIMPIO: tomra jabicho ibake
TARGET: tom estaba solo
-----
ORIGINAL: non aka akinra yoikanke
LIMPIO: non aka akinra yoikanke
TARGET: fuimos culpados
-----
ORIGINAL: moa joi riki
LIMPIO: moa joi riki
TARGET: ella está caminando
-----


***Creación de dataset***

In [12]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 5776
})


In [13]:
print(dataset[0])

{'source': 'jatian miabicho ika ikon', 'target': 'entonces estabas solo verdad'}


***Tokenización***

In [15]:
print("¿Existe nuevos_tokens?", 'nuevos_tokens' in globals())
print("¿Existe tokens_filtrados?", 'tokens_filtrados' in globals())
print("¿Existe df?", 'df' in globals())

¿Existe nuevos_tokens? True
¿Existe tokens_filtrados? False
¿Existe df? True


In [16]:
##Llamar al tokenizador de mBART50

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

print("Cantidad nuevos_tokens:", len(nuevos_tokens))

tokens_filtrados = [
    token
    for token in nuevos_tokens
    if token not in tokenizer.get_vocab()
]

print("Cantidad tokens_filtrados:", len(tokens_filtrados))

tokens_agregados = tokenizer.add_tokens(tokens_filtrados)

print("Tokens agregados:", tokens_agregados)

Cantidad nuevos_tokens: 146
Cantidad tokens_filtrados: 32
Tokens agregados: 32


In [17]:
##Probar tokenización

example = dataset[0]["source"]

print("Texto:")
print(example)

tokens = tokenizer(example)

print("\nTokens:")
print(tokens["input_ids"])

Texto:
jatian miabicho ika ikon

Tokens:
[250004, 79, 55283, 324, 14508, 3089, 134368, 28625, 2]


In [18]:
## Función para tokenizar todo el dataset

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [19]:
##Tokenizar todo el dataset

tokenized_dataset = dataset.map(tokenizar_ejemplo)

Map:   0%|          | 0/5776 [00:00<?, ? examples/s]

In [20]:
print(tokenized_dataset[0])

{'source': 'jatian miabicho ika ikon', 'target': 'entonces estabas solo verdad', 'input_ids': [250004, 79, 55283, 324, 14508, 3089, 134368, 28625, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'labels': [250004, 58333, 18724, 7, 2639, 36370, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [21]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

##Limpiar el dataset de columnas iniciales

tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

***Modelo***

In [22]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(
    "facebook/mbart-large-50-many-to-many-mmt"
)

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Embedding(250086, 1024)

In [23]:
##Configuración para el entrenamiento

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",              # dónde guardar modelo
    per_device_train_batch_size=1,       # batch pequeño (GPU segura)
    num_train_epochs=3,                  # Modelo ve 6 veces el dataset
    learning_rate=5e-5,                  # estándar para fine-tuning
    save_strategy="epoch",               # guarda cada epoch
    fp16=True                            # usa GPU eficientemente
)

In [24]:
##Creación del trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [25]:
#Entrenar el modelo
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Step,Training Loss
500,1.028300
1000,0.343600
1500,0.321100
2000,0.292700
2500,0.267100
3000,0.353300
3500,0.249000
4000,0.241800
4500,0.231100
5000,0.239800


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#sav

TrainOutput(global_step=17328, training_loss=0.17241423997412317, metrics={'train_runtime': 1324.8209, 'train_samples_per_second': 13.08, 'train_steps_per_second': 13.08, 'total_flos': 2347001538674688.0, 'train_loss': 0.17241423997412317, 'epoch': 3.0})

In [26]:
#Guardar el modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4/tokenizer.json')

In [46]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4


In [28]:
#Creación de la función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_length=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# VALIDACIÓN (val.csv)

In [29]:
#Cargar val

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [30]:
#Preprocesamiento de val
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)


In [31]:
#Ejemplos de los resultados

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", df_val.iloc[i]["target_clean"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: cabayoain ea neti atipana
REAL: puedo montar un caballo
PRED: puedo cocinar
-----
INPUT: tsoaki joa
REAL: quién ha venido
PRED: ha venido alguien
-----
INPUT: tsonbira yoiyamake
REAL: nadie respondió
PRED: nadie intervino
-----
INPUT: napomea ea meniwe
REAL: dame la mitad
PRED: denme la mitad
-----
INPUT: bakishra en oinkasai
REAL: espero verla mañana
PRED: espero mañana
-----


# Métricas para val.csv

In [32]:
#Preparar listas

preds = []
refs = []

for i in range(len(df_val)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = df_val.iloc[i]["target_clean"]

    preds.append(pred)
    refs.append([ref])

In [33]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")
print("BLEU:", bleu.compute(predictions=preds, references=refs))

BLEU: {'bleu': 0.17265227506389494, 'precisions': [0.4527027027027027, 0.23925925925925925, 0.15348101265822786, 0.06976744186046512], 'brevity_penalty': 0.9355671869032808, 'length_ratio': 0.9375565610859729, 'translation_length': 2072, 'reference_length': 2210}


In [34]:
#CHRF

chrf = evaluate.load("chrf")
print("ChrF:", chrf.compute(predictions=preds, references=refs))

ChrF: {'score': 42.17800082425554, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [35]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(df_val)):
    data.append({
        "src": df_val.iloc[i]["source"],  #ORIGINAL
        "mt": preds[i],
        "ref": df_val.iloc[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8, gpus=1)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

hparams.yaml:   0%|          | 0.00/437 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') th

COMET: -0.172438379826019


In [36]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: {'meteor': 0.3327659272335078}


In [37]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Promedio): 0.8349069157961002
BERTScore Recall (Promedio): 0.8317099265609752
BERTScore F1 (Promedio): 0.8330046036732164


In [39]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL:
# 1. learning_rate=3e-5 (Se reduce a 3e-5 en comparación con el valor inicial de 5e-5)
#    para realizar un ajuste fino más conservador y evitar el olvido catastrófico de los pesos
#    ya optimizados en la fase 1.
# 2. num_train_epochs = 3 (Se mantiene en 3 épocas para el refinamiento adicional).
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=3,
    learning_rate=3e-5,
    save_strategy="epoch"
)


In [40]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [41]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [47]:
from transformers import AutoModelForSeq2SeqLM
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_ashaninka_v4


Embedding(250086, 1024, padding_idx=1)

In [48]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [49]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
500,0.045000
1000,0.047600
1500,0.055600
2000,0.050600
2500,0.049700
3000,0.053400
3500,0.049900
4000,0.046600
4500,0.048600
5000,0.052200


Checkpoint destination directory ./results/checkpoint-5776 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}
Checkpoint destination directory ./results/checkpoint-11552 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-defa

TrainOutput(global_step=17328, training_loss=0.028240988046508746, metrics={'train_runtime': 1288.1862, 'train_samples_per_second': 13.451, 'train_steps_per_second': 13.451, 'total_flos': 2347001538674688.0, 'train_loss': 0.028240988046508746, 'epoch': 3.0})

In [50]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained")

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 200, 'early_stopping': True, 'num_beams': 5, 'forced_eos_token_id': 2}


('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained/sentencepiece.bpe.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/modelo_mbart_Ashaninka_v4_retrained/tokenizer.json')

In [52]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos

val_dataset = Dataset.from_pandas(df_val)

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: cabayoain ea neti atipana
REAL: puedo montar un caballo.
PRED: puedo comer algo
-----
INPUT: tsoaki joa
REAL: ¿quién ha venido?
PRED: quién vino
-----
INPUT: tsonbira yoiyamake
REAL: nadie respondió.
PRED: nadie intervino
-----
INPUT: napomea ea meniwe
REAL: dame la mitad.
PRED: denme la mitad
-----
INPUT: bakishra en oinkasai
REAL: espero verla mañana.
PRED: espero verla mañana
-----


In [53]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [54]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.10296187895271998,
 'precisions': [0.3902321083172147,
  0.1961367013372957,
  0.12836767036450078,
  0.06372549019607843],
 'brevity_penalty': 0.6508992654966054,
 'length_ratio': 0.699594046008119,
 'translation_length': 2068,
 'reference_length': 2956}

In [55]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 39.674489825182945, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [56]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.7221829660051087


In [57]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


METEOR (Validación - Reentrenado): {'meteor': 0.21057055594395915}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [58]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Validación - Reentrenado - Promedio): 0.7787814744621763
BERTScore Recall (Validación - Reentrenado - Promedio): 0.7721928472829327
BERTScore F1 (Validación - Reentrenado - Promedio): 0.7751455103921758


Revision de TEST.CSV

In [59]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Ashaninka/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [60]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [61]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 723
})


In [62]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [63]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.15561627568169936,
 'precisions': [0.43932853717026377,
  0.20851688693098386,
  0.12752721617418353,
  0.06046511627906977],
 'brevity_penalty': 0.9545428141430411,
 'length_ratio': 0.9555453712190651,
 'translation_length': 2085,
 'reference_length': 2182}

In [64]:
##CHRF
chrf.compute(predictions=preds, references=refs)

{'score': 41.59324933020317, 'char_order': 6, 'word_order': 0, 'beta': 2}

In [65]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.7527473932067718


In [66]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


METEOR: {'meteor': 0.3196134838389046}


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [67]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Promedio): 0.8301370068044913
BERTScore Recall (Promedio): 0.8267398558216965
BERTScore F1 (Promedio): 0.8281522874680463
